# Retrieval Augmented Generation (RAG)

In [ ]:
# in context learning:  give some example in prompt to AI to solve the problem
# RAF is a emerging property of GPT models
# RAG: give context and question as prompt to AI

# Generate multiple questions from user query , based on that pull data

In [37]:
# Load data from source, split and store in Vector Store with different collection name (except Wikipedia
# URL using web Scrapper
# PDF (Simple PDF)
# Doc file
# Text file
# Wikipedia on need basis ------------ at the time of quiry


# ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
# GoogleGenerativeAIEmbeddings(model="models/embedding-001")

True

# Load and Split data in chunks

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader, WebBaseLoader, CSVLoader, UnstructuredFileLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
#from langchain.retrievers import EnsembleRetriever
from dotenv import load_dotenv
load_dotenv()

chunk_size = 500
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_size
)


In [177]:
# URL - Data loading from given URL
url = 'https://www.flipkart.com/apple-macbook-air-m2-16-gb-256-gb-ssd-macos-sequoia-mc7x4hn-a/p/itmdc5308fa78421','https://www.flipkart.com/marq-flipkart-2025-1-5-ton-5-star-split-inverter-5-in-1-convertible-turbo-cool-technology-ac-white/p/itm0da4708578e88'
# We can also pass multiple URLs in the form of list of URLs
loader = WebBaseLoader(url)

docs = loader.load()

url_chunks = splitter.split_documents(docs)


In [178]:
# PDF, Docs, CSV and text file Loader
directory_path = "C:/Users/white/OneDrive/Desktop/docs"

all_documents = []

# For txt File
loader = DirectoryLoader(
directory_path,
glob="**/*.txt",  # This pattern includes all files
loader_cls = TextLoader,
show_progress=True, # Optional: Display a progress bar during loading
) 

all_documents.extend(loader.load())
print(len(all_documents))

# For PDF File
loader = DirectoryLoader(
directory_path,
glob="**/*.pdf",  # This pattern includes all files
loader_cls = PyPDFLoader,
show_progress=True, # Optional: Display a progress bar during loading
) 

all_documents.extend(loader.load())
print(len(all_documents))

# # For docx File  ---- Not working
# loader = DirectoryLoader(
# directory_path,
# glob="**/*.docx",  # Use UnstructuredFileLoader for .doc and .docx files
# loader_cls = UnstructuredFileLoader,
# show_progress=True, # Optional: Display a progress bar during loading
# ) 

# all_documents.extend(loader.load())
# print(len(all_documents))


# # For doc File  ---- Not working
# loader = DirectoryLoader(
# directory_path,
# glob="**/*.doc",  # Use UnstructuredFileLoader for .doc and .docx files
# loader_cls = UnstructuredFileLoader,
# show_progress=True, # Optional: Display a progress bar during loading
# ) 

# all_documents.extend(loader.load())
# print(len(all_documents))

# For PDF File
loader = DirectoryLoader(
directory_path,
glob="**/*.csv",  # This pattern includes all files
loader_cls = CSVLoader,
show_progress=True, # Optional: Display a progress bar during loading
) 

all_documents.extend(loader.load())
print(len(all_documents))

# for doc in all_documents:
#     print("Content:", doc.page_content[:50])
#     print("Metadata:", doc.metadata)
#     print()

docs_chunks = splitter.split_documents(all_documents)   


100%|██████████| 1/1 [00:00<00:00, 506.19it/s]


1


100%|██████████| 2/2 [00:01<00:00,  1.72it/s]


20


100%|██████████| 1/1 [00:00<00:00, 87.13it/s]

420


In [ ]:
# # Load data from Wiki on real time based on topic/query

# from langchain_community.retrievers import WikipediaRetriever

# # Initialize the retriever (optional: set language and top_k)
# retriever = WikipediaRetriever(top_k_results=2, lang="en")  # language can be change

# # Define your query
# # query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# # Get relevant Wikipedia documents
# docs = retriever.invoke(input('Ask your question: '))
# #print(docs)

# for i, text in enumerate(docs):
#     print(f'-----Result {i+1} -----\n')
#     print(text,'\n')

# Vector Store

In [179]:
vector_store_web = Chroma(
    embedding_function= GoogleGenerativeAIEmbeddings(model="models/embedding-001"),   # Embedding function
    persist_directory='test_chroma_db',  # Folder where you want to store db
    collection_name='web_sample'  # collection name
)

vector_store_doc = Chroma(
    embedding_function= GoogleGenerativeAIEmbeddings(model="models/embedding-001"),   # Embedding function
    persist_directory='test_chroma_db',  # Folder where you want to store db
    collection_name='doc_sample'  # collection name
)

vector_store_web.add_documents(url_chunks)
print("vector_store_web Size",len(vector_store_web.get()['ids']))

vector_store_doc.add_documents(docs_chunks)
print("vector_store_doc Size",len(vector_store_doc.get()['ids']))

vector_store_web Size 5307
vector_store_doc Size 2907


# Retrieval

In [189]:
# Set up the compressor using an LLM
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
compressor = LLMChainExtractor.from_llm(llm)


# Create the contextual compression retriever
retriever_web = ContextualCompressionRetriever(
    base_retriever= vector_store_web.as_retriever(search_kwargs={"k": 5}),
    base_compressor= LLMChainExtractor.from_llm(llm)
)

retriever_doc = ContextualCompressionRetriever(
    base_retriever= vector_store_doc.as_retriever(search_kwargs={"k": 5}),
    base_compressor= LLMChainExtractor.from_llm(llm)
)



In [190]:
parser = StrOutputParser()

prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context1}
      {context2}
      
      Question: {question}
    """,
    input_variables = ['context1','context2', 'question',]
)

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [191]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

retrieval_and_context = RunnableParallel(
    context1=retriever_web | RunnableLambda(format_docs), # Output will be docs from retriever_general
    context2=retriever_doc  | RunnableLambda(format_docs), # Output will be docs from retriever_technical
    question=RunnablePassthrough() # Pass the original question through
)

chain = retrieval_and_context | prompt | model | parser
result = chain.invoke(input('Ask your question'))
print(result)

Ask your question iphone rate?


I don't know.


In [ ]:

# Ensure you have your OpenAI API key set as an environment variable or passed explicitly.
genai.configure(api_key="add your key")

class CodeRAGSystem:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.documents = self._load_documents()
        self.chunks = self._chunk_documents()
        self.vectorstore = self._create_vectorstore()

    def _load_documents(self):
        """Loads the PDF document using PyMuPDFLoader."""
        loader = PyMuPDFLoader(self.pdf_path)  #
        return loader.load()

    def _chunk_documents(self):
        """Splits the loaded documents into smaller, manageable chunks."""
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)  #
        return text_splitter.split_documents(self.documents)

    def _create_vectorstore(self):
        """Generates embeddings for chunks and stores them in a FAISS vector store."""
        embeddings = genai.GenerativeModel("gemini-embedding-001")  # You can replace this with other embedding models
        db = FAISS.from_documents(self.chunks, embeddings)  #
        return db

    def query(self, question, k=5):
        """Retrieves relevant chunks and generates an answer using an LLM."""
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        qa = RetrievalQA.from_chain_type(llm = genai.GenerativeModel("gemini-2.5-flash"), chain_type="stuff", retriever=retriever)
        response = qa.run(question)
        return response

# Example usage
if __name__ == "__main__":
    pdf_file = "C:/Users/white/OneDrive/Desktop/Types_and_systems_of_farming-489.pdf"  # Replace with the path to your PDF
    rag_system = CodeRAGSystem(pdf_file)

    query = "Explain the function 'my_function' in the code."
    answer = rag_system.query(query)
    print(answer)

    # query = "Provide a code example of how to use the 'parse_data' method."
    # answer = rag_system.query(query)
    # print(answer)
